# Step 04. Which modules track clinical traits

One correlation per module per trait, then Benjamini–Hochberg across the whole table. With 52
modules and 15 traits that is 780 tests; uncorrected, roughly 39 would clear p < 0.05 by chance
alone.

The traits are read from `cohorts/clinical-traits.csv`, name, how to make a number of it, and
which antigen group it belongs to. It is committed beside the cohort matrices because changing it
changes every q-value in this notebook.

In [1]:
source("../src/paths.R")
suppressMessages(library(WGCNA))
w <- readRDS(art("wgcna_A.rds")); e <- readRDS(art("eigengenes_A.rds"))
X <- w$X; m <- w$meta; mods <- w$mods; ME <- e$ME

# The 15 traits are no longer a list typed into this cell. They come from
# cohorts/clinical-traits.csv, which also records each trait's TYPE (how to
# make a number of it) and its GROUP (which traits carry the same information
# and must therefore be held out together -- see the VarSelLCM section below).
spec <- read_traits()
stopifnot(all(spec$source_column %in% names(m)))

build_traits <- function(m, spec) {
  out <- data.frame(row.names = rownames(m))
  for (i in seq_len(nrow(spec))) {
    v <- m[[spec$source_column[i]]]
    out[[spec$name[i]]] <- switch(spec$type[i],
      numeric = as.numeric(as.character(v)),
      binary  = as.numeric(v == "Positive"),
      # Age enters as the INDEX of its five-year band, not as an invented exact
      # age. The bands are equally spaced, so this gives exactly the same
      # correlation a midpoint would -- it just does not claim to know anyone's age.
      ordinal = { lvl <- unique(v[!is.na(v) & v != ""])
                  lvl <- lvl[order(as.numeric(sub("-.*", "", lvl)))]
                  as.integer(factor(v, levels = lvl, ordered = TRUE)) },
      stop("unknown trait type '", spec$type[i], "' for ", spec$name[i]))
  }
  out
}
tr <- build_traits(m, spec)
stopifnot(ncol(tr) == nrow(spec))

r <- cor(ME, tr, use = "pairwise.complete.obs")
p <- corPvalueStudent(r, nrow(X))
q <- matrix(p.adjust(p, "BH"), nrow = nrow(p), dimnames = dimnames(p))
sprintf("%d modules x %d traits = %d tests, %d at FDR 5%%",
        nrow(q), ncol(q), length(q), sum(q < 0.05))

[1] "52 modules x 15 traits = 780 tests, 24 at FDR 5%"

In [2]:
h <- which(q < 0.05, arr.ind = TRUE)
res <- data.frame(module = sub("^ME", "", rownames(r)[h[, 1]]),
                  trait = colnames(r)[h[, 2]],
                  r = round(r[h], 2), q = signif(q[h], 2),
                  n_proteins = sapply(sub("^ME", "", rownames(r)[h[, 1]]),
                                      function(k) sum(mods == k)))
head(res[order(-abs(res$r)), ], 15)

,module,trait,r,q,n_proteins
,<chr>,<chr>,<dbl>,<dbl>,<int>
14,ivory,Sm_status,0.53,7.5e-05,14
10,darkmagenta,uPCR,0.50,3.4e-04,19
13,blue,Creatinine,0.49,4.2e-04,1213
12,black,Creatinine,-0.47,9.0e-04,150
7,bisque4,uPCR,0.46,9.0e-04,10
4,ivory,C3_level,-0.45,1.5e-03,14
24,ivory,Age_band,-0.45,1.6e-03,14
19,darkolivegreen,dsDNA_status,-0.43,2.6e-03,21
2,greenyellow,SLEDAI_2K,0.42,4.4e-03,78


## Interferon module

**The interferon module is the most clinically connected thing in the panel**, and it reproduces the
published result on this cohort. Fourteen proteins, nine associations surviving Benjamini-Hochberg (BH) correction across
780 tests:

| trait | r | q |
|---|---|---|
| anti-Sm | +0.53 | 7.5e-05 |
| C3 | −0.45 | 1.5e-03 |
| age band | −0.45 | 1.6e-03 |
| anti-Ro60 | +0.40 | 8.7e-03 |
| disease duration | −0.39 | 8.8e-03 |
| anti-dsDNA | +0.39 | 1.2e-02 |
| anti-RNP-A | +0.37 | 1.6e-02 |
| anti-RNP68 | +0.36 | 2.6e-02 |
| lymphocyte count | −0.35 | 3.4e-02 |

The source paper reports that positivity for antibodies against RNA-binding proteins, anti-Sm,
anti-Ro60, anti-RNP68, anti-RNP-A, went with increased interferon-stimulated proteins. All four
are here, and all four survive correction. Anti-Sm is the single strongest association in the
entire table.

Two things make this more than a coincidence of multiple testing. Anti-La is flat, r = +0.16,
q = 0.65, and it is not on the paper's list, so the one antibody that should *not* track is the one
that does not. And SLEDAI-2K just misses at q = 0.082, which is the expected behaviour of a
module tracking an immunological axis rather than a clinical severity score.

**Discount the large modules.** `blue` (1,213 proteins) correlating with creatinine and `black`
(150) with creatinine are eigenproteins over a large fraction of the panel, close to leading
principal components of the whole matrix, where correlating with something is nearly guaranteed. The
interferon module carries its associations on 14 proteins, which is why it is interpretable.

**Still open.** This is one fit on one cohort. The module's boundary has moved across earlier draws
of the same patients, so `modulePreservation` across B and C is what decides whether this table
describes biology or this sample.

## Which traits was it worth testing at all?

The table above tests every trait in `cohorts/clinical-traits.csv` against every module. That list
is a choice, and until now it was a choice typed into a code cell with no argument attached to
it. VarSelLCM turns it into a question the data answers: it fits a latent class model to the
clinical variables and selects, by BIC, which of them carry information about the patient
partition and which are noise.

**Why the groups in the CSV matter here.** Holding out one autoantibody does not hold out its
information. In cohort A, anti-Sm correlates +0.52 with anti-RNP-A and +0.35 with
anti-RNP-68, they are the same snRNP antigen system, and anti-Ro52 correlates +0.73 with
anti-Ro60. Drop anti-Sm alone and the snRNP signal walks straight back in through the other two.
So the hold-out below removes an entire antigen group at a time, which is what the `group` column
in `clinical-traits.csv` is for.

**This runs alongside the correlation table, not instead of it.** The BH-corrected table above is
the reported result; VarSelLCM is evidence about whether the trait list deserved its 15 entries.

In [3]:
# VarSelLCM is the ONE dependency this project cannot declare in
# endotypes-proteomics.yml -- there is no conda package for it. So it is
# installed here, from CRAN, into the active environment's R library.
#
# Guarded by requireNamespace(), so a second run of this notebook -- or a run
# under run_all.sh -- is a no-op and touches no network.
ensure_pkg("VarSelLCM")           # no conda package at all; see src/paths.R
suppressMessages(library(VarSelLCM))
packageVersion("VarSelLCM")

[1] ‘2.1.3.2’

In [4]:
set.seed(42)
GVALS  <- 2:4         # let BIC choose the number of latent classes
# nbcores = 1, NOT 4. VarSelCluster with nbcores > 1 gives each worker its own
# RNG stream, so set.seed() does not make it reproducible -- three runs at
# nbcores = 4 returned two different trait sets for this cohort, while three
# runs at nbcores = 1 were identical. Slower, and the only way the selection
# below is the same selection tomorrow.
NCORES <- 1

# VarSelLCM wants the binary traits as factors -- as 0/1 numerics it would model
# them as Gaussian, which for a two-point distribution is simply wrong.
clin <- tr
for (v in spec$name[spec$type == "binary"])
  clin[[v]] <- factor(clin[[v]], levels = c(0, 1), labels = c("Negative", "Positive"))
clin <- clin[, vapply(clin, function(x) length(unique(x[!is.na(x)])) > 1, logical(1)), drop = FALSE]

vs   <- VarSelCluster(clin, gvals = GVALS, vbleSelec = TRUE,
                      crit.varsel = "BIC", nbcores = NCORES)
kept <- names(clin)[vs@model@omega == 1]
cat(sprintf("%d latent classes; VarSelLCM keeps %d of %d traits\n\n",
            vs@model@g, length(kept), ncol(clin)))
print(data.frame(trait = names(clin),
                 group = spec$group[match(names(clin), spec$name)],
                 kept  = ifelse(names(clin) %in% kept, "kept", "-"),
                 row.names = NULL))

4 latent classes; VarSelLCM keeps 6 of 15 traits



              trait       group kept
1         SLEDAI_2K    activity kept
2          C3_level  complement    -
3          C4_level  complement    -
4    Duration_years     history    -
5  Lymphocyte_count haematology kept
6              uPCR       renal kept
7        Creatinine       renal kept
8         Sm_status       snRNP kept
9     RNP_68_status       snRNP    -
10     RNP_A_status       snRNP    -
11     Ro_52_status        RoLa    -
12     Ro_60_status        RoLa    -
13        La_status        RoLa    -
14     dsDNA_status       dsDNA    -
15         Age_band demographic kept


### Hold out analysis

Each row below refits with a each antigen group removed. If a group's information is genuinely
carried by that group alone, removing it should change which traits survive; if the selection is
unchanged, the information was redundant with what remains, which is the leak the snRNP grouping
was written to prevent.

In [5]:
groups <- unique(spec$group[spec$name %in% names(clin)])
holdout <- do.call(rbind, lapply(groups, function(g) {
  cols <- setdiff(names(clin), spec$name[spec$group == g])
  f <- VarSelCluster(clin[, cols, drop = FALSE], gvals = GVALS, vbleSelec = TRUE,
                     crit.varsel = "BIC", nbcores = NCORES)
  data.frame(held_out = g, n_traits = length(cols), classes = f@model@g,
             kept = paste(cols[f@model@omega == 1], collapse = ", "))
}))
holdout

held_out,n_traits,classes,kept
<chr>,<int>,<int>,<chr>
activity,14,3,"C3_level, C4_level, Lymphocyte_count, uPCR, Creatinine, Sm_status, RNP_68_status, Age_band"
complement,13,4,"SLEDAI_2K, Lymphocyte_count, uPCR, Creatinine, Sm_status, Age_band"
history,14,4,"SLEDAI_2K, Lymphocyte_count, uPCR, Creatinine, Sm_status, Age_band"
haematology,14,4,"SLEDAI_2K, C3_level, uPCR, Creatinine, Sm_status, Age_band"
renal,13,3,"SLEDAI_2K, C3_level, C4_level, Lymphocyte_count, Sm_status, Age_band"
snRNP,12,4,"SLEDAI_2K, C3_level, Lymphocyte_count, uPCR, Creatinine, Ro_52_status, Age_band"
RoLa,12,4,"SLEDAI_2K, Lymphocyte_count, uPCR, Creatinine, Sm_status, Age_band"
dsDNA,14,4,"SLEDAI_2K, C3_level, C4_level, Lymphocyte_count, uPCR, Creatinine"
demographic,14,3,"SLEDAI_2K, C3_level, C4_level, Lymphocyte_count, uPCR, Creatinine, dsDNA_status"


In [6]:
sig <- sub("^ME", "", rownames(q)[apply(q, 1, function(z) any(z < 0.05))])
saveRDS(sig, art("sig_modules_A.rds"))
write.csv(r, art("module_trait_r_A.csv")); write.csv(q, art("module_trait_q_A.csv"))
saveRDS(list(fit = vs, kept = kept, holdout = holdout, spec = spec),
        art("varsel_traits_A.rds"))
sprintf("%d of %d modules carry at least one association", length(sig), nrow(q))

[1] "12 of 52 modules carry at least one association"

## Where this was run

Rendered notebooks are committed, so each records the machine, the R and the
package versions that produced its output.


In [7]:
run_provenance()

run on   : Annes-MacBook-Pro-193.local ( Darwin 27.0.0 )
date     : 2026-09-25 12:03 EDT 
R        : R version 4.4.3 (2025-02-28) | x86_64-apple-darwin13.4.0 
R comes from: /Users/adeslatt/miniforge3/envs/endotypes-proteomics 
packages :
   WGCNA            1.74
   ComplexHeatmap   2.22.0
   sva              3.54.0
   VarSelLCM        2.1.3.2
   cluster          2.1.8.1
   fpc              2.2.15
   circlize         0.4.18
